In [1]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

In [2]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")
PDF_loader = PyMuPDFLoader(
    file_path="./data/3D_Printed_Home_Feasibility_Study_FINAL_2021_AHFC_Branded.pdf",
    mode="page"
)
docs = PDF_loader.load()

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size= 500,
    chunk_overlap = 50,
    separators=["\n\n", "\n", "。", "！", "？", " ", ""],
    length_function = len,
)
split_docs = splitter.split_documents(docs)


In [4]:
db_directory = "./Vectorstores/chroma_db_3d_homes"
vector_store = Chroma(
    collection_name="3D_disaster_housing",
    embedding_function=embeddings,
    persist_directory="./Vectorstores"
)

In [5]:
vector_store.add_documents(
    documents=split_docs,
    ids = ["id"+str(i) for i in range(1, len(split_docs)+1)]
)

query_text = "please tell me someting about 3D printing housing."
results = vector_store.similarity_search(query_text, k=10)


In [6]:
print(results[0])

page_content='where the construction is to take place, size and scale of the housing structures to be 
printed, and then, with respect to different printer options, such things as maximum 
build size, transportability, mobility, open source or proprietary construction material and 
cost (See Task 2 of Feasibility Study and PSU Design and Engineering Analysis). 
 
• Samples from three Alaska sites (Anchorage, Juneau and Fairbanks) were tested as' metadata={'creationdate': '2021-08-04T10:20:49-08:00', 'modDate': "D:20210806091240-08'00'", 'file_path': './data/3D_Printed_Home_Feasibility_Study_FINAL_2021_AHFC_Branded.pdf', 'page': 12, 'keywords': '', 'source': './data/3D_Printed_Home_Feasibility_Study_FINAL_2021_AHFC_Branded.pdf', 'format': 'PDF 1.6', 'total_pages': 216, 'author': '', 'creationDate': "D:20210804102049-08'00'", 'moddate': '2021-08-06T09:12:40-08:00', 'title': '', 'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 16.3 (Macintosh)', 'subject': '', 'trapped': '

In [7]:
vector_store.delete(["id"+str(i) for i in range(1,5)]) 